# SQLAlchemy with PostgreSQL 기초

이 노트북은 SQLAlchemy 2.x 방식으로 PostgreSQL에 연결하고 SQL 실행, Core CRUD, ORM 기초를 실습합니다.

SQLAlchemy는 SQL을 몰라도 되는 도구가 아닙니다. SQL 실행과 테이블/객체 작업을 Python 코드 안에서 일관된 방식으로 다루게 해주는 도구입니다.

## 1. 패키지와 접속 정보 준비

PostgreSQL 드라이버는 기존 강의와 같은 `psycopg`를 사용하고, SQLAlchemy URL은 `postgresql+psycopg://...` 형식으로 작성합니다.

In [ ]:
import os
from datetime import datetime
from decimal import Decimal

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import Column, DateTime, Integer, MetaData, Numeric, String, Table, create_engine, delete, func, insert, select, text, update
from sqlalchemy.orm import DeclarativeBase, Mapped, Session, mapped_column

load_dotenv()

DB_CONFIG = {
    "host": os.getenv("DB_HOST", "localhost"),
    "port": os.getenv("DB_PORT", "5432"),
    "database": os.getenv("DB_NAME", "deepagent_db"),
    "user": os.getenv("DB_USER", "admin"),
    "password": os.getenv("DB_PASSWORD", "admin123"),
}

DATABASE_URL = (
    f"postgresql+psycopg://{DB_CONFIG['user']}:{DB_CONFIG['password']}"
    f"@{DB_CONFIG['host']}:{DB_CONFIG['port']}/{DB_CONFIG['database']}"
)

engine = create_engine(DATABASE_URL, echo=False)
DATABASE_URL.replace(DB_CONFIG["password"], "****")

## 2. 연결 확인

`engine.connect()`로 연결을 얻고, `text()`로 SQL 문자열을 실행합니다.

In [ ]:
with engine.connect() as conn:
    row = conn.execute(text("SELECT version() AS version")).mappings().one()

row

## 3. text()로 SQL 직접 실행하기

SQLAlchemy에서도 SQL을 직접 작성할 수 있습니다. 값은 `:name` 형태의 바인딩 파라미터로 전달합니다.

In [ ]:
with engine.connect() as conn:
    rows = conn.execute(
        text("""
        SELECT
            table_name
        FROM information_schema.tables
        WHERE table_schema = :schema_name
        ORDER BY table_name
        LIMIT 10
        """),
        {"schema_name": "public"},
    ).mappings().all()

pd.DataFrame(rows)

## 4. SQLAlchemy Core로 테이블 정의하기

Core는 테이블과 SQL 문을 Python 객체로 표현합니다. ORM보다 SQL에 더 가까운 방식입니다.

In [ ]:
metadata = MetaData()

sa_students = Table(
    "sqlalchemy_students",
    metadata,
    Column("student_id", Integer, primary_key=True, autoincrement=True, comment="수강생을 식별하는 자동 증가 기본키"),
    Column("name", String(50), nullable=False, comment="수강생 이름"),
    Column("email", String(120), nullable=False, unique=True, comment="수강생 이메일, 중복 불가"),
    Column("score", Numeric(5, 2), comment="SQLAlchemy 실습용 점수"),
    Column("created_at", DateTime(timezone=True), nullable=False, server_default=func.now(), comment="수강생 등록 시각"),
    comment="SQLAlchemy Core 실습용 수강생 정보",
)

metadata.drop_all(engine, tables=[sa_students], checkfirst=True)
metadata.create_all(engine, tables=[sa_students])

print("sqlalchemy_students 테이블 준비 완료")

## 5. Core로 INSERT와 SELECT 실행하기

`engine.begin()` 블록은 정상 종료 시 자동으로 commit하고, 오류가 나면 rollback합니다.

In [ ]:
student_rows = [
    {"name": "김민준", "email": "sa_minjun@example.com", "score": Decimal("92.50")},
    {"name": "이서연", "email": "sa_seoyeon@example.com", "score": Decimal("85.00")},
    {"name": "박도윤", "email": "sa_doyun@example.com", "score": Decimal("78.00")},
]

with engine.begin() as conn:
    conn.execute(insert(sa_students), student_rows)

stmt = select(sa_students).order_by(sa_students.c.student_id)

with engine.connect() as conn:
    rows = conn.execute(stmt).mappings().all()

pd.DataFrame(rows)

## 6. Core로 조건 조회, 수정, 삭제하기

In [ ]:
minimum_score = Decimal("80.00")

stmt = (
    select(sa_students.c.name, sa_students.c.email, sa_students.c.score)
    .where(sa_students.c.score >= minimum_score)
    .order_by(sa_students.c.score.desc())
)

with engine.connect() as conn:
    rows = conn.execute(stmt).mappings().all()

pd.DataFrame(rows)

In [ ]:
with engine.begin() as conn:
    conn.execute(
        update(sa_students)
        .where(sa_students.c.email == "sa_minjun@example.com")
        .values(score=Decimal("95.00"))
    )
    conn.execute(delete(sa_students).where(sa_students.c.email == "sa_doyun@example.com"))

with engine.connect() as conn:
    rows = conn.execute(select(sa_students).order_by(sa_students.c.student_id)).mappings().all()

pd.DataFrame(rows)

## 7. ORM 기초 맛보기

ORM은 테이블을 Python 클래스로 표현합니다. 아래 예제는 같은 데이터베이스에 별도 ORM 실습 테이블을 만듭니다.

In [ ]:
class Base(DeclarativeBase):
    pass


class OrmStudent(Base):
    __tablename__ = "sqlalchemy_orm_students"
    __table_args__ = {"comment": "SQLAlchemy ORM 실습용 수강생 정보"}

    student_id: Mapped[int] = mapped_column(Integer, primary_key=True, autoincrement=True, comment="수강생 ID")
    name: Mapped[str] = mapped_column(String(50), nullable=False, comment="수강생 이름")
    email: Mapped[str] = mapped_column(String(120), nullable=False, unique=True, comment="수강생 이메일")
    score: Mapped[Decimal | None] = mapped_column(Numeric(5, 2), comment="점수")
    created_at: Mapped[datetime] = mapped_column(DateTime(timezone=True), server_default=func.now(), comment="등록 시각")


Base.metadata.drop_all(engine, tables=[OrmStudent.__table__], checkfirst=True)
Base.metadata.create_all(engine, tables=[OrmStudent.__table__])

with Session(engine) as session:
    session.add_all([
        OrmStudent(name="최하은", email="orm_haeun@example.com", score=Decimal("91.00")),
        OrmStudent(name="정지후", email="orm_jihu@example.com", score=Decimal("82.50")),
    ])
    session.commit()

with Session(engine) as session:
    students = session.scalars(select(OrmStudent).order_by(OrmStudent.score.desc())).all()

[{"name": student.name, "email": student.email, "score": student.score} for student in students]

## 정리

- `create_engine()`은 데이터베이스 연결을 만드는 출입구입니다.
- `text()`는 SQL 문자열을 안전하게 실행할 때 사용합니다.
- Core는 테이블과 SQL 문을 Python 객체로 표현합니다.
- ORM은 테이블을 Python 클래스로 표현하고 `Session`으로 객체를 저장/조회합니다.
- SQLAlchemy를 쓰더라도 SQL 구조와 관계형 데이터베이스 개념은 계속 중요합니다.